In [ ]:
from __future__ import (absolute_import, division,
                        print_function, unicode_literals)

import warnings
warnings.simplefilter('ignore')

# general purpose packages
import pandas as pd
import numpy as np
import os
import json
import time
import re
import csv
import subprocess
import sys

import scipy.stats as stats
import statsmodels.stats as smstats
from statsmodels.stats.multitest import multipletests

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from dotenv import load_dotenv
from pathlib import Path

from multiprocessing import Process, Manager, Pool
import multiprocessing
from functools import partial

from collections import Counter

import seaborn as sns; sns.set()

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
matplotlib.rcParams['backend'] = "Qt5Agg"
import matplotlib.ticker as ticker
from matplotlib.ticker import FuncFormatter

from IPython.display import display, Image

from adjustText import adjust_text
import builtins
%matplotlib inline

# for normalization
from sklearn.linear_model import QuantileRegressor

# for survival analysis
import sklearn
from sklearn import set_config

from statsmodels.regression.quantile_regression import QuantReg

# for working with yaml files
import ruamel.yaml

import itertools

%load_ext autoreload
%reload_ext autoreload
%autoreload 2
# this is important to be able to re-import the module after making modifications to the zavolab_pyutils code on Scicore

In [ ]:
def get_pvalue_star(pval, thr=0.05):
    if thr == 0.05:
        if pval < 0.001:
            return "***"
        elif pval < 0.01:
            return "**"
        elif pval < 0.05:
            return "*"
        else:
            return ""
    elif thr == 0.1:
        if pval < 0.001:
            return "***"
        elif pval < 0.01:
            return "**"
        elif pval < 0.1:
            return "*"
        else:
            return ""

In [ ]:
# 1. Load the environment variables
load_dotenv("TemperatureDependentCelegansAging_Spang.scicore.env",override=True)

# 2. Reconstruct the subdirs dictionary
subdirs = {
    "lab_group_dir": os.getenv("LAB_GROUP_DIR"),
    "raw_sequencing_data_dir": os.getenv("RAW_SEQUENCING_DATA_DIR"),
    "main_project_dir": os.getenv("MAIN_PROJECT_DIR"),
    "wf_dir": os.getenv("WF_DIR"),
    "UCSCtracks_dir": os.getenv("UCSC_TRACKS_DIR"),
    "UCSCtracks_trackfiles_dir": os.getenv("UCSC_TRACKFILES_DIR"),
    "UCSCtracks_trackhubs_dir": os.getenv("UCSC_TRACKHUBS_DIR"),
    "celegans_annotation_dir": os.getenv("CELEGANS_ANNOTATION_DIR"),
    "shared_project_dir": os.getenv("SHARED_PROJECT_DIR"),
    "temp_dir": os.getenv("TEMP_DIR"),
    "slurm_dir": os.getenv("SLURM_DIR"),
    "slurm_scripts_dir": os.getenv("SLURM_SCRIPTS_DIR"),
    "figures_dir": os.getenv("FIGURES_DIR"),
    "tables_dir": os.getenv("TABLES_DIR"),
    "fastq_dir": os.getenv("FASTQ_DIR"),
    "metadata_dir": os.getenv("METADATA_DIR"),
    "wf_runs_dir": os.getenv("WF_RUNS_DIR"),
    "configs_dir": os.getenv("CONFIGS_DIR"),
}

# 3. Reconstruct the file_paths dictionary
file_paths = {
    "celegans_genome_file": os.getenv("CELEGANS_GENOME_FILE"),
    "celegans_chrom_sizes_file": os.getenv("CELEGANS_CHROM_SIZES_FILE"),
    "celegans_annotation_file": os.getenv("CELEGANS_ANNOTATION_FILE"),
    "celegans_polyAsite_atlas": os.getenv("CELEGANS_POLYASITE_ATLAS"),
}

# 4. Safely create all subdirectories
# Using os.makedirs is highly preferred over os.system('mkdir -p')
# because it avoids opening a subshell and handles permissions gracefully in pure Python.
for path in subdirs.values():
    if path:  # Safety check to ensure the variable was actually found in the .env
        os.makedirs(path, exist_ok=True)

print("Environment loaded and directories verified.")

# Prepare start samples

In [ ]:
paths_to_look = [
subdirs['raw_sequencing_data_dir']+'20260228*', # these are the newest long RNA data, based on biotinylation, made with circularization protocol, they are single-end
]

fastq_file_paths = []
for path_ in paths_to_look:
    command = """find """+path_+""" -name '*.fastq.gz' > """+subdirs['temp_dir']+"""fastq_file_paths.tsv"""
    out = subprocess.check_output(command, shell=True)
    tmp = pd.read_csv(subdirs['temp_dir']+'fastq_file_paths.tsv',delimiter="\t",
                                   index_col=None,header=None)
    fastq_file_paths.append(tmp)
fastq_file_paths = pd.concat(fastq_file_paths).reset_index(drop=True)
fastq_file_paths.columns = ['path']

fastq_file_paths['sample'] = fastq_file_paths['path'].str.split('/',expand=True).iloc[:,-1].str.split('.',expand=True)[0]

In [ ]:
# Create symbolink link copies for all experimental fastq files

fastq_file_paths['start_file_path'] = subdirs['fastq_dir']+fastq_file_paths['sample']+'.fastq.gz'

# uncomment if need to re-generate, otherwise re-creation may invoke unnecessary Snakemake re-execution
for index, row in fastq_file_paths.iterrows():
    command = 'rm -f '+row['start_file_path']+' && ln -f -s '+row['path']+' '+row['start_file_path']
    out = subprocess.check_output(command, shell=True)

fastq_file_paths = fastq_file_paths.drop(['path'],axis=1).rename(columns={'start_file_path':'path'})

In [ ]:
# lanes should be aligned separately, but then merged together
fastq_file_paths['lane'] = fastq_file_paths.apply(lambda x:int(x['sample'].split('_L00')[1].split('_')[0]),1)
fastq_file_paths['merged_sample'] = fastq_file_paths.apply(lambda x:x['sample'].split('_')[0],1) # GFB... is in general consistent naming

start_samples = fastq_file_paths.copy()

In [ ]:
start_samples['fq1'] = start_samples['path']
start_samples['fq1_3p'] = 'AGATCGGAAGAG' # forcely put Illumina universal adapter

In [ ]:
WF_version = 'v1_0_0'

dir_path = subdirs['wf_runs_dir']+WF_version+'/'
command = 'mkdir -p '+dir_path
out = subprocess.check_output(command, shell=True)

start_samples.to_csv(dir_path+'start_samples.tsv', sep=str('\t'),header=True,index=None,quoting=csv.QUOTE_NONE)

In [ ]:
# all merged_samples should have TWO samples corresponding to two lanes:
start_samples['t']=1
start_samples.groupby(['merged_sample']).agg({'t':np.sum})

# Prepare .yaml config file and run WF

create conda environment with snakemake and install SLURM executor

run from the login node on HPC cluster:

```bash
conda create -c conda-forge -c bioconda -n snakemake snakemake
conda activate snakemake
pip install snakemake-executor-plugin-slurm
```

In [ ]:
organism = 'celegans'

gtf_chrs = pd.read_csv(file_paths[organism+'_annotation_file'],delimiter="\t",
                                   index_col=None,header=None,usecols = [0],skiprows=5)
chromosome_list = list(gtf_chrs[0].unique())

In [ ]:
# load default rule_config, modify it and save
WF_version = 'v1_0_0'
organism = 'celegans'

yaml = ruamel.yaml.YAML()
yaml.preserve_quotes = True
with open(subdirs['wf_dir']+'config.yaml') as f_read:
    data = yaml.load(f_read)

data['samples_file'] = subdirs['wf_runs_dir']+WF_version+'/start_samples.tsv'
data['output_dir'] = subdirs['wf_runs_dir']+WF_version+'/output/'
data['local_log'] = subdirs['wf_runs_dir']+WF_version+'/output/local_log/'
data['cluster_log'] = subdirs['wf_runs_dir']+WF_version+'/output/cluster_log/'

data['organism'] = organism
data['genome_file'] = file_paths[organism+'_genome_file']
data['gtf_file'] = file_paths[organism+'_annotation_file']

data['chromosomes'] = ' '.join(chromosome_list)

for dir_path in [data['output_dir'],data['local_log'],data['cluster_log']]:
    command = 'mkdir -p '+dir_path
    out = subprocess.check_output(command, shell=True)

with open(subdirs['wf_runs_dir']+WF_version+'/modified_config.yaml','w') as f_write:     
    yaml.dump(data, f_write)

# the following steps should be executed one after another
# uncomment the step to print the snakemake command below

# WF_step = "prepare-faster-se"
WF_step = "quantification-faster-se"

command = """snakemake \
--snakefile """+subdirs['wf_dir']+"""Snakefile-"""+WF_step+""" \
--scheduler greedy \
--configfile """+subdirs['wf_runs_dir']+WF_version+'/modified_config.yaml'+""" \
--printshellcmds \
--software-deployment-method conda apptainer \
--conda-frontend conda \
--apptainer-args "--bind """+subdirs['wf_dir']+','+subdirs['lab_group_dir']+','+subdirs['raw_sequencing_data_dir']+"""" \
--executor slurm \
--profile """+subdirs['wf_dir']+'profile'+""" \
--nolock \
-np"""

print(command)
# remember to run the command when "snakemake" conda env is activated!
# NOTE - there is an argument "-np" in the end - that results in a mock run, creating a DAG of jobs, to check the correctness of the planned run.
# To actually run, just delete the "-np" from the end of the command.

execute the command above from login node

snakemake automatically submits and monitors computational jobs to SLURM queue manager

The results of all workflow steps will be necessary to conduct further data analysis below, e.g. analyzing gene expression and alternative polyadenylation (APA)

## Define metadata labels and colors uniformly

In [ ]:
def get_metadata_with_labels_and_colors(input_df):
    sel_files = input_df.copy()
    sel_files["condition"] = sel_files.apply(
        lambda x: x["sample"].split("-")[2][:3], 1
    )
    sel_files["replicate"] = sel_files.apply(lambda x: x["sample"].split("-")[2][3:], 1)

    sel_files["sample_label_within_experiment"] = (
        sel_files["condition"].astype('str')+ ";"
        + sel_files["replicate"].astype('str')+ ";"
        + "L"+sel_files["lane"].astype('str')
    )

    cat = "condition"
    cat_order = ["23C", "OSC"]
    cat_palette = [
        "grey",
        "royalblue"
    ]
    cat_color_dict = {}
    for k, cat_val in enumerate(cat_order):
        cat_color_dict[cat_val] = cat_palette[k]
    sel_files[cat + ";color"] = sel_files[cat].map(cat_color_dict)

    sel_files = sel_files.sort_values(
        ["condition", "replicate", "merged_sample", "sample"]
    ).reset_index(drop=True)
    return sel_files

In [ ]:
# check how it will look:
get_metadata_with_labels_and_colors(start_samples.copy())

# Analysis of read mapping stats

In [ ]:
WF_version = "v1_0_0"
organism = "celegans"
dir_path = subdirs["wf_runs_dir"] + WF_version + "/"

os.system(
    """find """
    + dir_path
    + "output/mapping_stats/"
    + """ -name '*.mapping_stats.txt' > """
    + subdirs["temp_dir"]
    + """mapping_stats.files.txt"""
)
os.system(
    """find """
    + dir_path
    + "output/mapping_stats/"
    + """ -name '*.success' > """
    + subdirs["temp_dir"]
    + """mapping_stats.success.files.txt"""
)

mapping_stats_files = pd.read_csv(
    subdirs["temp_dir"] + "mapping_stats.files.txt",
    delimiter="\t",
    index_col=None,
    header=None,
)
mapping_stats_success_files = pd.read_csv(
    subdirs["temp_dir"] + "mapping_stats.success.files.txt",
    delimiter="\t",
    index_col=None,
    header=None,
)

mapping_stats_files["sample"] = mapping_stats_files.apply(
    lambda x: x[0].split("/")[-1].replace(".mapping_stats.txt", ""), 1
)
mapping_stats_success_files["sample"] = mapping_stats_success_files.apply(
    lambda x: x[0].split("/")[-1].replace(".success", ""), 1
)

mapping_stats_files = mapping_stats_files.loc[
    mapping_stats_files["sample"].isin(
        list(mapping_stats_success_files["sample"].unique())
    )
].reset_index(
    drop=True
)  # look only at files with success flag

start_samples = pd.read_csv(
    dir_path + "start_samples.tsv", delimiter="\t", index_col=None, header=0
)
metadata_df = get_metadata_with_labels_and_colors(start_samples.copy())

sel_files = pd.merge(
    mapping_stats_files,
    metadata_df.copy(),
    how="right",
    on=["sample"],
)
samples_list = list(sel_files["sample"])  # list of samples

In [ ]:
from zavolab_pyutils.parsing_workflow_outputs import (
    parse_mapping_stats
)

sub_sel_files = sel_files.copy()

i = 0
res = []
for index, row in sub_sel_files.iterrows():
    res_df = parse_mapping_stats(row[0],verbose=False)
    res_df["sample"] = row["sample"]
    res.append(res_df)
    if i % 2 == 0 and i != 0:
        print(f"{i} out of {len(sub_sel_files)} done")
    i = i + 1
mapping_stats_df = pd.concat(res).reset_index(drop=True)
numb_cols = list(mapping_stats_df.columns)[:3]
mapping_stats_df[numb_cols] = mapping_stats_df[numb_cols].astype('float')

mapping_stats_df["number of MM reads; genomic mapping"] = (
    mapping_stats_df["total number of mapped reads; genomic mapping"]
    - mapping_stats_df["number of UM reads; genomic mapping"]
)
numb_cols.append("number of MM reads; genomic mapping")

for elem in numb_cols:
    mapping_stats_df[elem + "; mln"] = np.round(mapping_stats_df[elem] / 10**6, 2)

mapping_stats_df['perc_of_UM'] = mapping_stats_df['number of UM reads; genomic mapping']/mapping_stats_df['total number of input reads']*100
mapping_stats_df['perc_of_MM'] = mapping_stats_df['number of MM reads; genomic mapping']/mapping_stats_df['total number of input reads']*100

# add metadata for visualization
mapping_stats_df = pd.merge(mapping_stats_df,sel_files, how="left", on="sample")

In [ ]:
features = [
    "total number of input reads; mln",
    "number of UM reads; genomic mapping; mln",
    "number of MM reads; genomic mapping; mln",
]
palette = ["black", "green", "orange"]

sns.set(font_scale=1)
sns.set_style("white")
fig, axes = plt.subplots(1, 1, sharey=False, sharex=True, figsize=(4, 4))

for i, feature in enumerate(features):

    x_feature, y_feature = feature, "sample_label_within_experiment"

    ax = sns.pointplot(
        data=mapping_stats_df, x=x_feature, y=y_feature, color=palette[i], label=feature
    )
    ax.legend(bbox_to_anchor=(1.05, 1.0), loc=2, borderaxespad=0.0, title="", ncols=1)
    # ax.set_yticklabels(mapping_stats_df['sample_label_within_experiment'])
    ax.tick_params(left=True, bottom=True)
ax.set(ylabel="", xlabel="# reads, mln")
out = subprocess.check_output(
    "mkdir -p " + subdirs["figures_dir"] + "mapping_stats/", shell=True
)
fig.savefig(
    subdirs["figures_dir"] + "mapping_stats/read_mapping.read_numbers.png",
    bbox_inches="tight",
    dpi=300,
)
fig.savefig(
    subdirs["figures_dir"] + "mapping_stats/read_mapping.read_numbers.pdf",
    bbox_inches="tight",
    dpi=300,
)

In [ ]:
features = ["perc_of_UM","perc_of_MM"]
palette = ["green","orange"]

sns.set(font_scale=1)
sns.set_style("white")
fig, axes = plt.subplots(1, 1, sharey=False, sharex=True, figsize=(4, 4))

for i, feature in enumerate(features):

    x_feature, y_feature = feature, "sample_label_within_experiment"

    ax = sns.pointplot(
        data=mapping_stats_df, x=x_feature, y=y_feature, color=palette[i], label=feature
    )
    ax.legend(bbox_to_anchor=(1.05, 1.0), loc=2, borderaxespad=0.0, title="", ncols=1)
    ax.tick_params(left=True, bottom=True)
ax.set(ylabel="", xlabel="% of mapped reads")
out = subprocess.check_output(
    "mkdir -p " + subdirs["figures_dir"] + "mapping_stats/", shell=True
)
fig.savefig(
    subdirs["figures_dir"] + "mapping_stats/read_mapping.perc_of_UM.png",
    bbox_inches="tight",
    dpi=300,
)
fig.savefig(
    subdirs["figures_dir"] + "mapping_stats/read_mapping.perc_of_UM.pdf",
    bbox_inches="tight",
    dpi=300,
)

# Analysis of gene expression

In [ ]:
WF_version = "v1_0_0"
organism = "celegans"

os.system("""find """+subdirs['wf_runs_dir']+WF_version+'/output/gene_expression_quantification/'+\
          organism+'/FeatureCounts_standard/'+""" -name '*.standard.txt' > """+
          subdirs['temp_dir']+"""standard_FeatureCounts.files.txt""")

In [ ]:
entity_col = "merged_sample" # should be a unique index for the metadata table

exonic_segments_FeatureCounts_files = pd.read_csv(subdirs['temp_dir']+'standard_FeatureCounts.files.txt',delimiter="\t",
                                   index_col=None,header=None)
exonic_segments_FeatureCounts_files[entity_col] = exonic_segments_FeatureCounts_files.apply(lambda x:x[0].split('/')[-1].replace('.standard.txt',''),1)

exonic_segments_FeatureCounts_files = exonic_segments_FeatureCounts_files.rename(columns={0:'path'})

start_samples = pd.read_csv(subdirs['wf_runs_dir']+WF_version+'/'+'start_samples.tsv',delimiter="\t",index_col=None,header=0)
metadata_df = get_metadata_with_labels_and_colors(start_samples.copy())
metadata_df = metadata_df[[entity_col,'condition','replicate','condition;color']].drop_duplicates().reset_index(drop=True)
metadata_df = pd.merge(metadata_df,exonic_segments_FeatureCounts_files,how='inner',on=entity_col)

In [ ]:
i=0
for index,row in metadata_df.iterrows():
    tmp = pd.read_csv(row['path'],delimiter="\t",index_col=None,header=0,skiprows=1)
    cols = list(tmp.columns)
    tmp = tmp.rename(columns={cols[-1]:row[entity_col]})
    if i==0:
        res = tmp.copy()
    else:
        res = pd.merge(res,tmp,how='outer',on=cols[:-1])
    if i%5==0:
        print(str(i)+' done')
    i=i+1

In [ ]:
sel_metadata = metadata_df.copy()
sel_metadata = sel_metadata.rename(columns={entity_col:'sample'}) # rename to sample for consistency with downstream analysis
samples_list = list(sel_metadata['sample'])

index_cols = ['Geneid', 'Chr', 'Start', 'End', 'Strand', 'Length']
cur_res = res[index_cols+samples_list].reset_index(drop=True)
cur_res[samples_list] = cur_res[samples_list].fillna(0).astype('int')
cur_res = cur_res.rename(columns={'Geneid':'gene_id'}) # for easier correspondence with annotation file
index_cols = ['gene_id', 'Chr', 'Start', 'End', 'Strand', 'Length']

#### Deseq style analysis

In [ ]:
from zavolab_pyutils.read_count_data_analysis import (
    apply_deseq2_normalization
)

norm_cur_res_df, sfs_df = apply_deseq2_normalization(cur_res,sel_metadata.copy(),pseudocount=1)
norm_cur_res_df = pd.concat([cur_res[index_cols],norm_cur_res_df],axis=1) # add index cols

In [ ]:
from zavolab_pyutils.visualization import (
    plot_size_factors,
)

plot_size_factors(sfs_df,savefig_path=subdirs['figures_dir']+'bulk_RNAseq/diagnostic_plots/library_size_vs_SF.png')

In [ ]:
log2_norm_cur_res_df = norm_cur_res_df.copy() # make a log2 version for PCA and may be smth else
log2_norm_cur_res_df[samples_list] = np.log2(log2_norm_cur_res_df[samples_list]) # now we have normalized counts with respect to library size

In [ ]:
from zavolab_pyutils.annotation import (
    parse_gtf_attributes_into_pd_dataframes,
)

gtf_df, genes_df, exons_df = parse_gtf_attributes_into_pd_dataframes(file_paths[organism+'_annotation_file'],
                                                                     input_skiprows=0,
                                                                     gene_type_field = "gene_biotype", # Ensembl naming convention
                                                                     extract_exon_number = False, # we don't have exon numbers in this annotation
                                                                     extract_gene_name_in_exons = False, # we don't have it in Ensembl
                                                                     verbose=False,
                                                                    )

In [ ]:
# Run PCA

from zavolab_pyutils.visualization import (
    pca_plot
)

log2_norm_cur_res_df['m'] = log2_norm_cur_res_df[samples_list].mean(axis=1)
PCA_UMAP_subset = log2_norm_cur_res_df.loc[log2_norm_cur_res_df['m']>log2_norm_cur_res_df['m'].quantile(0.4)].copy().reset_index(drop=True)
print(f"used {len(PCA_UMAP_subset)} genes for PCA analysis")

savefig_path = (
    subdirs['figures_dir']+'bulk_RNAseq/PCA_gene_expression.Deseq_norm_counts.png'
)

hue_column = 'condition'
hue_df = sel_metadata.drop_duplicates(hue_column).reset_index(drop=True)
palette = list(hue_df['condition;color'])
hue_order = list(hue_df[hue_column])

pca_plot(
    PCA_UMAP_subset,
    samples_list,
    sel_metadata,
    hue_column,
    savefig_path,
    sns_color_palette = palette,
    hue_order = hue_order,
    calculate_permanova_R2=True,
    add_text_labels_for_samples=True,
    text_label_column_for_samples='replicate',
    text_label_size_for_samples=8,
    s_param=40,
    figsize=(4.2,4.2),
)

In [ ]:
test_df = pd.merge(genes_df[['gene_id','gene_name','gene_type']].drop_duplicates(),PCA_UMAP_subset,how='inner',on='gene_id')
test_df['t'] = 1
gr = test_df.groupby('gene_type').agg({'t':np.sum}).reset_index()
gr['%'] = np.round(gr['t']/gr['t'].sum()*100,2)
gr.sort_values('t',ascending=False)

In [ ]:
test_df.loc[test_df['gene_type']=='ncRNA'].sort_values('m',ascending=False).head()

##### DE with Deseq2

In [ ]:
# run differential analysis
# conda activate deseq

raw_matrix = cur_res.copy()  # prepare input for Deseq2
raw_matrix[samples_list] = np.round(raw_matrix[samples_list]).astype("int")

list_of_output_tables = []
command = ""

exclude_replicates = []

CtrlNonCtrl_condition = ["23C","OSC"]

CtrlCond,NonCtrlCond = CtrlNonCtrl_condition
sub_sel_metadata = sel_metadata[["sample", "condition"]]

sub_sel_metadata["EXP_CTL"] = sub_sel_metadata.apply(
    lambda x: "EXP" if x["condition"] == NonCtrlCond else "CTL", 1
)
index_cols = ["gene_id"]
sub_samples_list = list(sub_sel_metadata["sample"])

raw_matrix = raw_matrix[index_cols + sub_samples_list].reset_index(drop=True)
raw_matrix[sub_samples_list] = (
    raw_matrix[sub_samples_list].fillna(0).astype("int")
)
raw_matrix = (
    raw_matrix[["gene_id"] + sub_samples_list]
    .drop_duplicates("gene_id")
    .reset_index(drop=True)
)  # to double-check that there are no duplicate entries

prefix_path = (
    subdirs["tables_dir"]
    + "gene_expression/"
    + NonCtrlCond + "_vs_" + CtrlCond
    + "/"
)

out = subprocess.check_output(
    "mkdir -p " + prefix_path,
    shell=True,
)

counts_file_path = prefix_path + "counts.tsv"
metadata_file_path = prefix_path + "metadata.tsv"
raw_matrix.to_csv(
    counts_file_path,
    sep=str("\t"),
    header=True,
    index=None,
    quoting=csv.QUOTE_NONE,
)

# default Deseq2 model, no confounders
metadata_cols = ["EXP_CTL"] 
for col in ['confounder_1','confounder_2','confounder_3']:
    sub_sel_metadata[col] = 0
    metadata_cols.append(col)
sub_sel_metadata[["sample"] + metadata_cols].to_csv(
    metadata_file_path,
    sep=str("\t"),
    header=True,
    index=None,
    quoting=csv.QUOTE_NONE,
)

outdir_path = prefix_path
out = subprocess.check_output("mkdir -p " + outdir_path, shell=True)
command = (
    command
    + """Rscript """
    + subdirs["wf_dir"]
    + "scripts/run_deseq2.r"
    + " "
    + counts_file_path
    + " "
    + metadata_file_path
    + " EXP "
    + outdir_path
    + " fake_gene"
    + " no_confounders"
    + " 0" # use lfc=0 as null hypothesis
    + " FALSE; " # Don't use apeglm-shrinked log2FC values, for better visualization
)
list_of_output_tables.append(
    [NonCtrlCond, CtrlCond, outdir_path + "DE_table.tsv"]
)
print(command)